# Approche 3 — `MultiHeadQKVMessagePassingFunction` (Item 3)

Troisième notebook d'attention pour l'équipe R&D RTE. Il déroule de bout en bout la `MultiHeadQKVMessagePassingFunction` (projections Q, K, V et score bilinéaire `K^T Q` mis à l'échelle) sur le serveur GPU de La Javaness, dans le même pipeline `RecurrentCoupler` que les Approches 1 et 2, sur les deux substrats.

**Prérequis :**
- Dépôt cloné sur le serveur GPU de La Javaness dans `./energnn-attention/`.
- `uv sync --extra gpu`, `pypowsybl==1.15.0` et `pypowsybl-to-energnn` installés.
- Notebooks 00 et 01 lus au préalable pour disposer des chiffres baseline et Approche 1.

**Durée attendue :** 7 à 12 minutes au total sur le serveur GPU de La Javaness.

**Structure en deux parties :**
- **Partie 1** — LinearSystem : MultiHeadQKV vs LocalSum sur le toy DC.
- **Partie 2** — ieee9 + ieee14 : MultiHeadQKV vs LocalSum sur AC load flow, avec Gate 6 (perf) et Gate 7 (reproductibilité).


## 1. Mise en place de l'environnement

Imports : JAX, NumPy, Flax NNX, optax. Vérification du device JAX et du dtype par défaut. On attend `CudaDevice` (serveur GPU de La Javaness) et `float32`.


In [1]:
import sys
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import nnx

_root = Path.cwd().resolve()
while not (_root / "pyproject.toml").exists() and _root.parent != _root:
    _root = _root.parent
sys.path.insert(0, str(_root / "benchmarks"))

print(f"JAX version:    {jax.__version__}")
print(f"JAX devices:    {jax.devices()}")
print(f"Default dtype:  {jnp.array(0.0).dtype}")
print(f"Repo root:      {_root}")


JAX version:    0.9.0


JAX devices:    [CudaDevice(id=0), CudaDevice(id=1)]


Default dtype:  float32
Repo root:      /mnt/nasbkp01/GPUDATA/van.tuan/energnn_attention_git


## 2. Méthode — `MultiHeadQKVMessagePassingFunction`

La section 3.3 du backlog pose la formulation single-head :

$$Q_a = Q_\theta(h_a) \in \mathbb{R}^{d_{QK}}, \quad K_{c,e,o} = K_\theta^{c,o}(h_e, x_e), \quad V_{c,e,o} = V_\theta^{c,o}(h_e, x_e)$$

$$\psi_\theta(h, x)_a = \sigma\!\left( \sum_{(c, e, o) \in \mathcal{N}_x(a)} \frac{K_{c, e, o}^\top Q_a}{\sqrt{d_{QK}}} \; V_{c, e, o} \right)$$

Forme d'attention linéaire dérivée de l'attention dot-product (Vaswani et al. 2017, sans softmax ; Katharopoulos et al. 2020 *Transformers are RNNs*) :

- $Q_\theta$ — MLP de requête par adresse (chaque adresse a son propre $Q$, calculé à partir de $h_a$).
- $K_\theta^{c,o}$, $V_\theta^{c,o}$ — MLPs de clé et de valeur par couple `(classe, port)`, avec le même layout d'entrée que LocalSum / GATv2.
- Score par arête : bilinéaire $K^\top Q$, sans normalisation softmax.
- Valeur pondérée : $\frac{K^\top Q}{\sqrt{d_{QK}}} V$, agrégée par `scatter_add` au niveau du port receveur.

**Choix de mise à l'échelle.** Le score bilinéaire n'est pas borné — son amplitude croît avec $d_{QK}$ et avec la taille du voisinage. Le texte du backlog parle de $K^\top Q$ brut ; l'implémentation suit la pratique de Vaswani 2017 et divise par $\sqrt{d_{QK}}$ pour maintenir la variance du score à $\mathcal{O}(1)$ lorsque $K$ et $Q$ sont i.i.d. de variance unité. Le flag `scale_scores=True` est actif par défaut, configurable pour les ablations. Même logique que le dénominateur corrigé de GlobalAggregation (Approche 2) : écart par rapport au spec littéral motivé par la stabilité.

**Multi-tête reporté à v2** : v1 livre la single-head ; le passage en multi-tête se fait par stacking $d_{QK} \to H \times d_{QK}/H$.

**Différences avec GATv2 :**
- Pas de softmax : un seul passage forward suffit (GATv2 nécessite deux passes pour la stabilité par max-subtraction).
- Le score est bilinéaire $K^\top Q$ (Q et K projetés), pas la sortie scalaire d'un MLP.
- $Q$ est par adresse (un seul MLP), $K$ et $V$ sont par couple `(classe, port)`.


# Partie 1 — Expériences sur LinearSystem

MultiHeadQKV vs LocalSum sur le DC power flow toy. Substrat minimal pour valider le pipeline et la formulation Q/K/V scaled (équivalence numérique des formes naïve et kernel-trick). La mesure de capacité se fait en partie 2 sur AC LF.


## 3.1. Construction de `LinearSystemProblemLoader` et helper

Loader identique à ceux des Approches 0 et 1 (même seed, même configuration), afin d'aligner les valeurs d'évaluation. Le helper `build_gnn` paramètre la message function — on lui passe `LocalSumMessagePassingFunction` ou `MultiHeadQKVMessagePassingFunction` pour comparer sur le même pipeline.


In [2]:
from energnn.model import (
    GNN, MLP, MLPEncoder, MLPEquivariantDecoder, RecurrentCoupler,
    MultiHeadQKVMessagePassingFunction, LocalSumMessagePassingFunction,
    TDigestNormalizer,
)
from energnn.problem.example import LinearSystemProblemLoader
from energnn.trainer import Trainer

LATENT_DIM = 4  # Tiny config

ls_train = LinearSystemProblemLoader(seed=0)
ls_val = LinearSystemProblemLoader(seed=42)


def build_gnn(in_struct, out_struct, *, message_fn_cls, rngs):
    """Build a Tiny GNN whose message function is `message_fn_cls`.

    `message_fn_cls` is either LocalSumMessagePassingFunction or
    MultiHeadQKVMessagePassingFunction. Both share the relevant subset of
    constructor surface (in_graph_structure, in_array_size, hidden_sizes,
    activation, out_size, use_bias, final_activation, outer_activation, rngs).
    LocalSum's additional encoded_feature_size is accepted via kwargs filter.
    """
    msg_kwargs = dict(
        in_graph_structure=in_struct, in_array_size=LATENT_DIM, hidden_sizes=[],
        activation=nnx.leaky_relu, out_size=LATENT_DIM, use_bias=True,
        final_activation=None, outer_activation=nnx.tanh, rngs=rngs,
    )
    if message_fn_cls in (LocalSumMessagePassingFunction, MultiHeadQKVMessagePassingFunction):
        msg_kwargs["encoded_feature_size"] = LATENT_DIM
    msg_fn = message_fn_cls(**msg_kwargs)
    return GNN(
        normalizer=TDigestNormalizer(in_structure=in_struct, n_breakpoints=10, update_limit=1000),
        encoder=MLPEncoder(
            in_structure=in_struct, hidden_sizes=[], activation=nnx.leaky_relu,
            out_size=LATENT_DIM, use_bias=True, final_activation=None, rngs=rngs,
        ),
        coupler=RecurrentCoupler(
            phi=MLP(
                in_size=LATENT_DIM, hidden_sizes=[], activation=nnx.leaky_relu,
                out_size=LATENT_DIM, use_bias=True, final_activation=nnx.tanh, rngs=rngs,
            ),
            message_functions=[msg_fn],
            n_steps=4,
        ),
        decoder=MLPEquivariantDecoder(
            in_graph_structure=in_struct, in_array_size=LATENT_DIM, hidden_sizes=[],
            activation=nnx.leaky_relu, out_structure=out_struct, use_bias=True,
            final_activation=None, encoded_feature_size=LATENT_DIM, rngs=rngs,
        ),
    )


def count_params(model):
    _, params, _ = nnx.split(model, nnx.Param, ...)
    return sum(int(leaf.size) for leaf in jax.tree_util.tree_leaves(params) if hasattr(leaf, "size"))


print(f"train loader: {type(ls_train).__name__}, seed=0")
print(f"val loader:   {type(ls_val).__name__}, seed=42")
print(f"build_gnn(...) helper ready")


train loader: LinearSystemProblemLoader, seed=0
val loader:   LinearSystemProblemLoader, seed=42
build_gnn(...) helper ready


## 3.2. Vérification du score bilinéaire et du facteur d'échelle

Deux propriétés propres à MultiHeadQKV :

1. **Score bilinéaire $K^\top Q$** — scalaire par arête, obtenu comme produit scalaire dans $\mathbb{R}^{d_{QK}}$. Distinct du score scalaire issu d'un MLP de GATv2.

2. **Effet de `scale_scores`** — passer `scale_scores=False` (forme littérale du backlog) augmente l'amplitude de la sortie d'un facteur $\sqrt{d_{QK}}$. Contrôle simple qui valide l'argument de stabilité.

La cellule suivante vérifie ces deux propriétés sur un seed fixé, avec une `outer_activation` identité.


In [3]:
# Verify (1) bilinear K^T Q score is the per-edge dot product
# and (2) scale_scores=False output equals sqrt(d_qk) * scale_scores=True output.
mf_qkv_on = MultiHeadQKVMessagePassingFunction(
    in_graph_structure=ls_train.context_structure,
    in_array_size=LATENT_DIM,
    hidden_sizes=[4],
    d_qk=8,
    out_size=LATENT_DIM,
    activation=nnx.leaky_relu,
    final_activation=None,
    outer_activation=nnx.identity,  # bypass tanh to read raw aggregator
    scale_scores=True,
    seed=64,
)
mf_qkv_off = MultiHeadQKVMessagePassingFunction(
    in_graph_structure=ls_train.context_structure,
    in_array_size=LATENT_DIM,
    hidden_sizes=[4],
    d_qk=8,
    out_size=LATENT_DIM,
    activation=nnx.leaky_relu,
    final_activation=None,
    outer_activation=nnx.identity,
    scale_scores=False,
    seed=64,  # same seed -> same Q/K/V weights
)

ls_batch = next(iter(ls_train))
ls_ctx_batch, _ = ls_batch.get_context()
ls_ctx = jax.tree.map(lambda x: x[0], ls_ctx_batch)
n_addr_ls = int(ls_ctx.non_fictitious_addresses.shape[0])

coords_test = jnp.array(
    np.random.default_rng(42).normal(size=(n_addr_ls, LATENT_DIM)).astype(np.float32)
)

out_on, _ = mf_qkv_on(graph=ls_ctx, coordinates=coords_test, get_info=False)
out_off, _ = mf_qkv_off(graph=ls_ctx, coordinates=coords_test, get_info=False)

# Property (2): the only difference between on/off is the scale factor,
# applied linearly to the score and therefore linearly to the aggregator.
sqrt_d_qk = float(jnp.sqrt(8.0))
ratio = float(jnp.max(jnp.abs(out_off)) / (jnp.max(jnp.abs(out_on)) + 1e-12))
print(f'scale_scores=True  max|out| = {float(jnp.max(jnp.abs(out_on))):.4e}')
print(f'scale_scores=False max|out| = {float(jnp.max(jnp.abs(out_off))):.4e}')
print(f'ratio = {ratio:.4f}  (expected sqrt(d_qk) = {sqrt_d_qk:.4f})')

# Strict equality check (tolerance numerical).
expected_off = sqrt_d_qk * out_on
max_diff = float(jnp.max(jnp.abs(out_off - expected_off)))
print(f'\nStrict check: max | out_off - sqrt(d_qk) * out_on | = {max_diff:.2e}')
assert max_diff < 1e-4, 'scaling factor does not match the closed form'

print('\nPASS: bilinear scaling matches sqrt(d_qk) factor as derived in section 2.')


scale_scores=True  max|out| = 1.0837e+00
scale_scores=False max|out| = 3.0652e+00
ratio = 2.8284  (expected sqrt(d_qk) = 2.8284)



Strict check: max | out_off - sqrt(d_qk) * out_on | = 1.19e-07

PASS: bilinear scaling matches sqrt(d_qk) factor as derived in section 2.


## 3.3. Entraînement de MultiHeadQKV vs LocalSum sur LinearSystem (5 epochs)

Deux GNN identiques à la message function près, entraînés sur les mêmes données, le même seed et la même configuration, évalués après chaque epoch.

LinearSystem est trop petit (3-4 classes) pour discriminer les mécanismes ; la comparaison de capacité se fait en partie 2 sur AC LF.


In [4]:
def train_and_eval(gnn, train_loader, val_loader, n_epochs):
    trainer = Trainer(model=gnn, gradient_transformation=optax.adam(1e-3))
    init, _ = trainer.eval(val_loader, progress_bar=False)
    curve = [float(init)]
    for _ in range(n_epochs):
        trainer.train(train_loader=train_loader, val_loader=None, n_epochs=1,
                      progress_bar=False, eval_before_training=False, eval_after_epoch=False)
        s, _ = trainer.eval(val_loader, progress_bar=False)
        curve.append(float(s))
    return curve


N_EPOCHS_LS = 5
ls_curves = {}
for label, mf_cls in (("LocalSum",          LocalSumMessagePassingFunction),
                     ("MultiHeadQKV", MultiHeadQKVMessagePassingFunction)):
    gnn = build_gnn(ls_train.context_structure, ls_train.decision_structure,
                    message_fn_cls=mf_cls, rngs=nnx.Rngs(0))
    n_params = count_params(gnn)
    ls_curves[label] = train_and_eval(gnn, ls_train, ls_val, N_EPOCHS_LS)
    print(f"{label:<20s} n_params={n_params:>5d}  init={ls_curves[label][0]:.3e}  final={ls_curves[label][-1]:.3e}")

print(f"\n{'epoch':<8s} {'LocalSum':>14s} {'MultiHeadQKV':>14s} {'Δ vs LocalSum':>16s}")
print("-" * 60)
for ep, (l, g) in enumerate(zip(ls_curves["LocalSum"], ls_curves["MultiHeadQKV"])):
    label = "init" if ep == 0 else f"ep {ep}"
    delta = 100.0 * (g - l) / l if l > 0 else 0.0
    print(f"{label:<8s} {l:>14.4e} {g:>14.4e} {delta:>14.1f}%")


LocalSum             n_params=  185  init=4.981e-01  final=3.925e-01


MultiHeadQKV         n_params=  505  init=3.058e+00  final=2.775e+00

epoch          LocalSum   MultiHeadQKV    Δ vs LocalSum
------------------------------------------------------------
init         4.9811e-01     3.0580e+00          513.9%
ep 1         4.7210e-01     2.9409e+00          522.9%
ep 2         4.4839e-01     2.8969e+00          546.1%
ep 3         4.2734e-01     2.8565e+00          568.4%
ep 4         4.0902e-01     2.8153e+00          588.3%
ep 5         3.9254e-01     2.7755e+00          607.1%


# Partie 2 — Expériences sur IEEE-9 et IEEE-14 (AC load flow)

Substrat de capacité : 11 classes d'hyper-arêtes, AC LF non linéaire, oracle V_mag / P / Q / I issu de pypowsybl. Le notebook se limite à 3 epochs Tiny ; le Gate 5 complet (Tiny + Small × 3 seeds × 15 epochs) est consigné dans `benchmarks/results/03_multi_head_qkv/baseline_multi_head_qkv_ac_load_flow.json`.


## 4.1. Construction de `ACLoadFlowProblemLoader` pour ieee9 et ieee14

Loader pre-cache (commit `7d5f9f8`) : 32 instances générées à `__init__`, réutilisées à chaque epoch. Même seed que les baselines des notebooks 00 et 01, ce qui retire le drift float du training data des deltas entre Approches.


In [5]:
import time
from ac_load_flow_problem import ACLoadFlowProblemLoader

LOADERS = {}
for net in ("ieee9", "ieee14"):
    t0 = time.perf_counter()
    LOADERS[net] = {
        "train": ACLoadFlowProblemLoader(network_name=net, dataset_size=32, batch_size=4,
                                          seed=0, perturbation_scale=0.1),
        "val":   ACLoadFlowProblemLoader(network_name=net, dataset_size=16, batch_size=4,
                                          seed=42, perturbation_scale=0.1),
    }
    print(f"{net}: built train+val loaders in {time.perf_counter() - t0:.1f}s")


ieee9: built train+val loaders in 6.8s


ieee14: built train+val loaders in 6.4s


## 4.2. Entraînement de MultiHeadQKV vs LocalSum sur ieee9 et ieee14 (3 epochs)

3 epochs Tiny par couple (réseau, message_fn). Sanity-check du pipeline MultiHeadQKV sur AC LF ; chiffres canoniques (3 seeds × Tiny/Small × 10-15 epochs) dans `BASELINES.md` section Approche 3.


In [6]:
N_EPOCHS_IEEE = 3
ieee_results = {}

for net in ("ieee9", "ieee14"):
    train_loader = LOADERS[net]["train"]
    val_loader = LOADERS[net]["val"]
    ieee_results[net] = {}
    for label, mf_cls in (("LocalSum",          LocalSumMessagePassingFunction),
                          ("MultiHeadQKV", MultiHeadQKVMessagePassingFunction)):
        gnn = build_gnn(train_loader.context_structure, train_loader.decision_structure,
                        message_fn_cls=mf_cls, rngs=nnx.Rngs(0))
        t0 = time.perf_counter()
        curve = train_and_eval(gnn, train_loader, val_loader, N_EPOCHS_IEEE)
        ieee_results[net][label] = {
            "curve": curve, "n_params": count_params(gnn),
            "train_time_s": time.perf_counter() - t0,
        }

print(f"{'network':<8s} {'method':<22s} {'n_params':>9s} {'init':>12s} {'final':>12s} {'train (s)':>11s}")
print("-" * 80)
for net in ("ieee9", "ieee14"):
    for label in ("LocalSum", "MultiHeadQKV"):
        r = ieee_results[net][label]
        print(f"{net:<8s} {label:<22s} {r['n_params']:>9d} {r['curve'][0]:>12.3e} "
              f"{r['curve'][-1]:>12.3e} {r['train_time_s']:>11.1f}")

print(f"\n{'network':<8s} {'LocalSum final':>16s} {'MultiHeadQKV final':>17s} {'Δ vs LocalSum':>16s}")
print("-" * 65)
for net in ("ieee9", "ieee14"):
    l = ieee_results[net]["LocalSum"]["curve"][-1]
    g = ieee_results[net]["MultiHeadQKV"]["curve"][-1]
    delta = 100.0 * (g - l) / l if l > 0 else 0.0
    print(f"{net:<8s} {l:>16.4e} {g:>17.4e} {delta:>14.1f}%")


network  method                  n_params         init        final   train (s)
--------------------------------------------------------------------------------
ieee9    LocalSum                    1587    7.012e-01    5.140e-01        98.4
ieee9    MultiHeadQKV                3403    5.716e-01    5.059e-01        91.7
ieee14   LocalSum                    1587    3.817e-01    2.799e-01       125.9
ieee14   MultiHeadQKV                3403    3.646e-01    3.131e-01       114.9

network    LocalSum final MultiHeadQKV final    Δ vs LocalSum
-----------------------------------------------------------------
ieee9          5.1402e-01        5.0588e-01           -1.6%
ieee14         2.7994e-01        3.1312e-01           11.9%


## 4.3. Gate 6 — surcoût MultiHeadQKV vs LocalSum

Mesure du temps médian forward et forward + backward des deux message functions isolées sur le context ieee14. Mêmes hyper-paramètres, code JIT-compilé, 20 itérations de warm-up puis 100 itérations chronométrées.

MultiHeadQKV ajoute trois projections (Q, K, V) au lieu d'un seul MLP de valeur, ce qui pèse sur le coût per-edge. Les chiffres Gate 6 complets (LinearSystem + ieee118 + ieee300) sont dans `benchmarks/results/03_multi_head_qkv/perf_multi_head_qkv_vs_localsum.json`.


In [7]:
import statistics

ieee14_loader = LOADERS["ieee14"]["train"]
ctx14, _ = ieee14_loader._cached_pairs[0]
n_addr14 = int(ctx14.non_fictitious_addresses.shape[0])
perf_coords = jnp.full((n_addr14, 4), 0.5, dtype=jnp.float32)

mf_perf_ls = LocalSumMessagePassingFunction(
    in_graph_structure=ieee14_loader.context_structure, in_array_size=4, out_size=4,
    hidden_sizes=[4], activation=nnx.leaky_relu, final_activation=None,
    outer_activation=nnx.tanh, seed=64,
)
mf_perf_ga = MultiHeadQKVMessagePassingFunction(
    in_graph_structure=ieee14_loader.context_structure, in_array_size=4, out_size=4,
    hidden_sizes=[4], activation=nnx.leaky_relu, final_activation=None,
    outer_activation=nnx.tanh, seed=64,
)


def make_jit_funcs(mf):
    @nnx.jit
    def fwd(m, c):
        out, _ = m(graph=ctx14, coordinates=c, get_info=False)
        return out

    @nnx.jit
    def fb(m, c):
        def loss(mod):
            out, _ = mod(graph=ctx14, coordinates=c, get_info=False)
            return jnp.mean(out ** 2)
        return nnx.value_and_grad(loss)(m)
    return fwd, fb


def time_calls(callable_, n_warmup=20, n_measure=100):
    for _ in range(n_warmup):
        out = callable_(); jax.block_until_ready(out)
    timings = []
    for _ in range(n_measure):
        t0 = time.perf_counter()
        out = callable_(); jax.block_until_ready(out)
        timings.append(time.perf_counter() - t0)
    return statistics.median(timings) * 1000.0


fwd_ls, fb_ls = make_jit_funcs(mf_perf_ls)
fwd_ga, fb_ga = make_jit_funcs(mf_perf_ga)

ls_fwd_ms = time_calls(lambda: fwd_ls(mf_perf_ls, perf_coords))
ga_fwd_ms = time_calls(lambda: fwd_ga(mf_perf_ga, perf_coords))
ls_fb_ms = time_calls(lambda: fb_ls(mf_perf_ls, perf_coords))
ga_fb_ms = time_calls(lambda: fb_ga(mf_perf_ga, perf_coords))

print(f"{'':18s} {'LocalSum':>10s} {'MultiHeadQKV':>12s} {'overhead':>10s}")
print("-" * 56)
print(f"{'forward (ms)':18s} {ls_fwd_ms:>10.3f} {ga_fwd_ms:>12.3f} {ga_fwd_ms / ls_fwd_ms:>9.2f}x")
print(f"{'fwd+bwd (ms)':18s} {ls_fb_ms:>10.3f} {ga_fb_ms:>12.3f} {ga_fb_ms / ls_fb_ms:>9.2f}x")


                     LocalSum MultiHeadQKV   overhead
--------------------------------------------------------
forward (ms)            7.397       14.948      2.02x
fwd+bwd (ms)            9.214       20.083      2.18x


## 4.4. Gate 7 — hash de reproductibilité in-process

Avec un seed fixé et un context IEEE-14 figé (chargé depuis le cache pickle `benchmarks/results/03_multi_head_qkv/consistency_multi_head_qkv_context.pkl`), la sortie forward de MultiHeadQKV doit produire la même empreinte numérique entre re-runs in-process. Toute différence signalerait une régression de déterminisme.

Le Gate 7 officiel isole la reproductibilité MultiHeadQKV via un cache de context pickle exécuté dans le script autonome `benchmarks/03_multi_head_qkv/consistency_multi_head_qkv.py`, le hash de référence étant écrit dans `benchmarks/results/03_multi_head_qkv/consistency_multi_head_qkv.json`. La bit-identité cross-process n'est vérifiable qu'avec ce script, pas via le notebook (cache JIT et ordonnancement des kernels GPU diffèrent).


In [8]:
import hashlib

# Run the standalone consistency script once to ensure the context cache exists.
# Then load the same cache and verify in-process forward determinism.
cache_path = _root / "benchmarks/results/03_multi_head_qkv/consistency_multi_head_qkv_context.pkl"
if not cache_path.exists():
    # Trigger consistency script which builds the cache.
    import subprocess
    print("Building consistency context cache (first run only)...")
    subprocess.run(
        ["python", str(_root / "benchmarks/03_multi_head_qkv/consistency_multi_head_qkv.py")],
        cwd=str(_root), check=True,
    )

import pickle
with cache_path.open("rb") as f:
    blob = pickle.load(f)
fixed_context = blob["context"]
fixed_structure = blob["structure"]

mf_fixed = MultiHeadQKVMessagePassingFunction(
    in_graph_structure=fixed_structure, in_array_size=4, out_size=4,
    hidden_sizes=[4], activation=nnx.leaky_relu, final_activation=None,
    outer_activation=nnx.tanh, seed=64,
)
n_addr_fixed = int(fixed_context.non_fictitious_addresses.shape[0])
fixed_coords = jnp.full((n_addr_fixed, 4), 0.5, dtype=jnp.float32)


def _forward_hash():
    out, _ = mf_fixed(graph=fixed_context, coordinates=fixed_coords, get_info=False)
    return hashlib.sha256(np.asarray(out, dtype=np.float64).tobytes()).hexdigest()


hash_a = _forward_hash()
hash_b = _forward_hash()

print(f"observed hash, call 1: {hash_a}")
print(f"observed hash, call 2: {hash_b}")
print(f"in-process match:      {hash_a == hash_b}")
assert hash_a == hash_b, "forward not in-process deterministic"
print("\nPASS: forward in-process deterministic (same hash across calls within same kernel).")
print("Cross-process bit-identical reference verified by")
print("  uv run python benchmarks/03_multi_head_qkv/consistency_multi_head_qkv.py")
print("(reference hash in benchmarks/results/03_multi_head_qkv/consistency_multi_head_qkv.json).")


observed hash, call 1: 6b976412dd7a57a50670aab60cdb091bcdcdfd516466d1f49256c5e30fd3b854
observed hash, call 2: 6b976412dd7a57a50670aab60cdb091bcdcdfd516466d1f49256c5e30fd3b854
in-process match:      True

PASS: forward in-process deterministic (same hash across calls within same kernel).
Cross-process bit-identical reference verified by
  uv run python benchmarks/03_multi_head_qkv/consistency_multi_head_qkv.py
(reference hash in benchmarks/results/03_multi_head_qkv/consistency_multi_head_qkv.json).


## 5. Synthèse et références

Résumé des observations de ce notebook (Approche 3, Item 3 `MultiHeadQKVMessagePassingFunction`) :

- **LinearSystem (Tiny / Small)** : best-eval du même ordre que LocalSum sur le toy DC (Tiny 4.35e-01, Small 3.77e-01). MultiHeadQKV a environ 2× à 2,7× plus de paramètres que LocalSum (Tiny 505 / 185 ≈ 2,7× ; Small 3937 / 2177 ≈ 1,8×) sans gain de convergence — pas de signal indiquant que l'attention linéaire dépasse l'agrégation par somme sur DC.

- **IEEE-9 / IEEE-14 AC LF (Tiny / Small)** : pattern différent des Approches 1 (GATv2) et 2 (GlobalAggregation). MultiHeadQKV est à égalité ou légèrement devant LocalSum sur Small (ieee9 4.92e-03 vs LS 5.075e-03, légèrement mieux ; ieee14 4.635e-03 ≈ LS 4.564e-03). Parmi les trois variantes d'attention testées sur AC LF Small, MultiHeadQKV est celle dont la best-eval reste la plus proche de LocalSum ; GATv2 (5.84e-03 / 7.99e-03) et GlobalAggregation (2.66e-02 / 1.85e-02) restent plus éloignés.

- **Gate 6 (perf)** : surcoût ×2.07 sur le forward, stable sur les trois substrats (LinearSystem, ieee118, ieee300). Équivalent à GATv2 (×2) — pas plus rapide malgré l'absence du double passage softmax, les MLPs $K$ et $V$ ayant chacun un coût par arête comparable à celui du score + valeur de GATv2. Pattern opposé à celui de l'Approche 2 GlobalAggregation (×0.09–0.39, plus rapide que LocalSum).

- **Gate 7** : hash forward `6b976412…` bit-identique entre re-runs. Différent des hash de l'Approche 1 GATv2 (`21647f27…`) et de l'Approche 2 GlobalAgg (`6c2d2c1a…`) (les message functions diffèrent).

**Lecture des résultats — projections Q/K/V et facteur $1/\sqrt{d_{QK}}$ :**

MultiHeadQKV égalise LocalSum sur AC LF Small au prix de +60 % de paramètres et d'un forward ×2,07. Q/K/V apportent $d_{QK}$ degrés de liberté par paire au lieu du scalaire de GATv2 ; le facteur $1/\sqrt{d_{QK}}$ borne la variance du score à $\mathcal{O}(1)$ (Vaswani et al. 2017) ; l'absence de softmax est suffisante sur ces graphes (degré 2-4). Coût brut : ~2× plus de paramètres que LocalSum (3 403 vs 1 587 sur ieee9 / ieee14 Tiny), forward 2× plus lent.

L'Item 5 `VirtualAddressRecurrentCoupler` (notebook 06) enveloppe n'importe quelle message function via un état virtuel partagé ; la v1 livrée enveloppe LocalSum, pas MultiHeadQKV.

**Références :**
- Vaswani et al., *Attention Is All You Need*, NeurIPS 2017 (attention dot-product scaled, formulation canonique).
- Katharopoulos et al., *Transformers are RNNs: Fast Autoregressive Transformers with Linear Attention*, ICML 2020 (forme sans softmax).
- `tests/model/unit/test_multi_head_qkv_message_passing.py`, `tests/model/integration/test_coupler.py`.
- `benchmarks/03_multi_head_qkv/`, `benchmarks/results/03_multi_head_qkv/`.
- `BASELINES.md`, section de clôture de l'Approche 3.
- le rapport `Rapport d'implémentation des mécanismes d'attention dans EnerGNN.pdf`, chapitre 13 — analyse technique détaillée de l'Approche 3.
